# Forgetting Ledger — ResNet-18 + BatchNorm on Split CIFAR-10 (Google Colab)

1. *Runtime → Change runtime type → T4 GPU*.
2. Only if the repository is private: add a Colab secret **`GH_TOKEN`** (key icon) that can read `Basil-Mohammad/forgetting-ledger`.
3. Run the cells in order. Outputs and checkpoints live on Google Drive: after a disconnect, re-run all cells (you may skip the smoke test) and every job resumes from its last checkpoint.
4. Expected time: about 36 min per seed on a T4. When everything is finished run the last cell and download `flgr_results_cifar.zip` from `MyDrive/forgetting-ledger/`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/forgetting-ledger'
RUN_ROOT, DATA_ROOT, REPO_DIR = f'{BASE}/runs', '/content/data', '/content/forgetting-ledger'
LIMIT_H = 1000
import os; os.makedirs(RUN_ROOT, exist_ok=True)

In [ ]:
import os, subprocess
try:
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None   # public repository: no token needed
url = f"https://{tok}@github.com/Basil-Mohammad/forgetting-ledger.git" if tok else "https://github.com/Basil-Mohammad/forgetting-ledger.git"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
subprocess.run(["pip", "-q", "install", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
import torch
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NO GPU")

In [ ]:
# Smoke test on this GPU: the whole pipeline at a tiny scale (a few minutes). Run once before the real jobs.
import subprocess, sys, time
t0 = time.time()
r = subprocess.run([sys.executable, "scripts/run.py", *['all', 'configs/c10_resnet_bn.yaml', 'seed=99', 'train_per_task=320', 'first_task_train=640', 'train.first_task_epochs=1', 'train.epochs=1', 'probe_per_class=4', 'ledger.tracin_checkpoints=3', 'interventions.reps=1', 'interventions.removal_fracs=[0.1]', 'interventions.lds_subsets=2', 'interventions.surgery_targets=1', 'interventions.param_fracs=[0.1]'], f"out_root={RUN_ROOT}/../_smoke", f"data_root={DATA_ROOT}"],
                   cwd=REPO_DIR, capture_output=True, text=True, timeout=3600)
print(r.stdout[-4000:], r.stderr[-4000:])
assert r.returncode == 0, "smoke test failed - send the output above"
print(f"smoke test OK in {(time.time() - t0) / 60:.1f} min", flush=True)

In [ ]:
# One entry per job: "<command> <config> [overrides...]". Every job is resumable; finished steps are skipped.
JOBS = [f"all configs/c10_resnet_bn.yaml seed={s}" for s in range(10)]
JOBS = [j.split() for j in JOBS]
print(len(JOBS), "jobs")

In [ ]:
# Runs the jobs, one per GPU in parallel (Kaggle "T4 x2": two at a time). Re-running this cell after a
# disconnect resumes every job from its last checkpoint. Per-job logs: <RUN_ROOT>/../logs/.
import subprocess, sys, time, os, threading, queue, torch
LOGS = f"{RUN_ROOT}/../logs"; os.makedirs(LOGS, exist_ok=True)
NGPU = max(1, torch.cuda.device_count())
JOB_TIMEOUT_H = 3.0         # a job that runs longer than this is stopped (it resumes when the cell is re-run)
q = queue.Queue()
for j in JOBS:
    q.put(j)
T_START = time.time()
def worker(gpu):
    while True:
        if (time.time() - T_START) / 3600 > LIMIT_H:
            print(f"[gpu{gpu}] time budget reached - save and continue in a new session", flush=True); return
        try:
            job = q.get_nowait()
        except queue.Empty:
            return
        name = "_".join(job[1:]).replace("/", "_").replace("=", "-")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        t0 = time.time()
        print(f"[gpu{gpu}] >>> {' '.join(job)}", flush=True)
        with open(f"{LOGS}/{name}.log", "a") as f:
            try:
                rc = subprocess.call([sys.executable, "scripts/run.py", *job, f"out_root={RUN_ROOT}", f"data_root={DATA_ROOT}"],
                                     cwd=REPO_DIR, stdout=f, stderr=subprocess.STDOUT, env=env, timeout=JOB_TIMEOUT_H * 3600)
            except subprocess.TimeoutExpired:
                rc = "timeout"
        print(f"[gpu{gpu}] <<< exit {rc} after {(time.time() - t0) / 60:.1f} min: {' '.join(job)}", flush=True)
        if rc != 0:
            print(open(f"{LOGS}/{name}.log").read()[-3000:], flush=True)
        else:
            print("   " + open(f"{LOGS}/{name}.log").read().strip().splitlines()[-1][:200], flush=True)
threads = [threading.Thread(target=worker, args=(g,)) for g in range(NGPU)]
[t.start() for t in threads]; [t.join() for t in threads]
print("all workers finished")

In [ ]:
# Progress overview: forgetting and ledger completeness of every finished tracked run.
import json, glob, os
for d in sorted(glob.glob(f"{RUN_ROOT}/*-s*")):
    f = f"{d}/metrics.jsonl"
    m = [json.loads(l) for l in open(f)] if os.path.exists(f) else []
    t1 = [x for x in m if x.get("task") == 1]
    steps = sorted(os.listdir(f"{d}/interventions")) if os.path.isdir(f"{d}/interventions") else []
    if t1:
        a0 = [x for x in m if x.get("task") == 0][0]["task_acc"][0]
        print(os.path.basename(d), f"| forgetting {100 * (a0 - t1[0]['task_acc'][0]):.1f} pp",
              f"| completeness {100 * (t1[0]['completeness'] or 0):.2f} % (tv {100 * (t1[0]['completeness_tv'] or 0):.2f} %)",
              f"| evals/step {t1[0].get('evals_per_step') or 0:.1f} capped {100 * (t1[0].get('capped_frac') or 0):.0f} %",
              "| interventions:", [s.split('_')[0] for s in steps])
    else:
        print(os.path.basename(d), "| tracked run not finished yet")

In [ ]:
# Compact results for the paper (drops model snapshots, resumable states and the per-parameter arrays that
# the paper analysis does not need). Download the printed zip file and send it back.
import os, glob, shutil, subprocess, torch
DST = "/tmp/flgr_slim"
DROP = {"param", "learn_param", "unit_of", "layer_of"}
shutil.rmtree(DST, ignore_errors=True)
for run in sorted(glob.glob(f"{RUN_ROOT}/*")):
    for root, dirs, files in os.walk(run):
        dirs[:] = [d for d in dirs if d not in ("snapshots", "state", "adam_tmp")]
        for f in files:
            src = os.path.join(root, f); dst = os.path.join(DST, os.path.relpath(src, RUN_ROOT))
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if f.endswith(".pt") and "/ledger/" in src:
                d = torch.load(src, map_location="cpu", weights_only=False)
                torch.save({k: v for k, v in d.items() if k not in DROP}, dst)
            else:
                shutil.copy2(src, dst)
out = f"{RUN_ROOT}/../flgr_results_cifar.zip"
if os.path.exists(out):
    os.remove(out)
subprocess.run(f"cd {DST} && zip -qr {out} .", shell=True, check=True)
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")